# 받아쓴 대화를 GPT로 요약하기

4번 스크립트로 받아쓴 대화록(CSV)을 GPT에 넣어 핵심 내용을 요약한다.

> 이 계정의 OpenAI 정식 키는 크레딧이 없어서, 채팅을 지원하는 **OpenRouter**로 gpt-4o를 호출한다.
> OpenAI에 크레딧이 있다면 `base_url` 을 지우고 `OPENAI_DIRECT_KEY`, `model="gpt-4o"` 로 바꾸면 된다.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
# OpenRouter 키(sk-or-v1...)로 gpt-4o 채팅을 호출한다.
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key, base_url="https://openrouter.ai/api/v1")
MODEL = "openai/gpt-4o"

In [ ]:
# 받아쓴 CSV(start:end:text)에서 텍스트만 모아 대화록 문자열로 만든다.
csv_path = "./audio/싼기타_비싼기타.csv"

lines = []
with open(csv_path, encoding="utf-8") as f:
    next(f)  # 헤더(start:end:text) 건너뛰기
    for line in f:
        parts = line.rstrip("\n").split(":", 2)  # 텍스트에 ':'가 있어도 안전하게
        if len(parts) == 3:
            lines.append(parts[2].strip())

transcript = "\n".join(lines)
print(f"대화록 {len(transcript)}자")
print(transcript[:200], "...")

In [ ]:
# GPT에 줄 지시문(system_prompt): 대화록 + 요약 요청
system_prompt = (
    "다음은 음성에서 받아쓴 대화 녹취록입니다. "
    "핵심 내용을 3~5개의 불릿으로 한국어로 요약해 주세요.\n\n"
    f"{transcript}"
)

In [ ]:
# OpenAI API를 사용해 요약 결과 생성
response = client.chat.completions.create(
    model=MODEL,
    temperature=0.1,
    messages=[
        {"role": "system", "content": system_prompt},
    ],
)

summary = response.choices[0].message.content
summary = summary.strip()  # 좌우 공백 제거

print(summary)

In [ ]:
# 요약 결과를 파일로 저장
output_path = "./audio/싼기타_비싼기타_summary.txt"
with open(output_path, "w", encoding="utf-8") as f:
    f.write(summary)

print("저장:", output_path)